In [ ]:
from transformers import (
    AutoTokenizer,
    pipeline,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset
import pandas as pd

In [ ]:
from datasets import Dataset

MODEL_NAME = "roberta-base"
DATA_DIR = "/kaggle/input/datasets/ababeiraul/tweets"

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

hf_train = Dataset.from_pandas(train_df)
hf_test = Dataset.from_pandas(test_df)

print(test_df.head(1))

In [ ]:
print(len(test_df))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

token = tokenizer.tokenize(hf_train["text"][0])

print(token)

In [ ]:
id2label = {
    0: "REAL",
    1: "FAKE"
}

label2id = {
    "REAL": 0,
    "FAKE": 1
}

def preprocess_text(examples):
    return tokenizer([" ".join(x) for x in examples["text"]])

tokenized_df_train = hf_train.map(
    preprocess_text,
    batched = True,
    num_proc = 4
)

tokenized_df_test = hf_test.map(
    preprocess_text,
    batched = True,
    num_proc = 4
)

In [ ]:
def true_process_function(examples):
    return tokenizer(
        examples["text"],
        truncation = True,
        padding=False,
        max_length=128
    )

tokenized_train = tokenized_df_train.map(
    true_process_function,
    batched = True,
    num_proc = 4
)

tokenized_test = tokenized_df_test.map(
    true_process_function,
    batched = True,
    num_proc = 4
)

In [ ]:
# tokenized_test = tokenized_test.rename_column("target", "labels")
tokenized_train = tokenized_train.rename_column("target", "labels")

In [ ]:
print(tokenized_test.column_names)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
!pip install -q evaluate

In [ ]:
import evaluate 
import numpy as np

f1_score = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):

    logits, labels = eval_pred
    predictions = np.argmax(logits, axis = 1)

    f1 = f1_score.compute(
        predictions=predictions,
        references=labels,
        average="weighted" # accounts imbalance in the dataset
    )

    accuracy = accuracy_metric.compute(
        predictions = predictions,
        references = labels,
    )

    return {
        "f1": f1["f1"],
        "accuracy": accuracy["accuracy"]
    }




In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/tweets_model",

    # base hyperparams
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    fp16 = True,
    weight_decay = 0.01,

    # evaluation and saving
    eval_strategy = "epoch",
    save_strategy = "epoch",
    save_total_limit = 1, # only save one checkpoint
    load_best_model_at_end = True, # gives us the best checkpoint of the model
    metric_for_best_model = "f1",

    # logging
    logging_steps = 50, # log every 50 steps
    report_to = "none"
)

In [ ]:
split = tokenized_train.train_test_split(test_size = 0.1, seed = 42)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = split["train"],
    eval_dataset = split["test"],
    data_collator = data_collator,
    compute_metrics = compute_metrics
)

In [ ]:
train_results = trainer.train()

In [ ]:
print(train_results)

In [ ]:

raw_predicts = trainer.predict(tokenized_test);

predicted_classes = np.argmax(raw_predicts.predictions, axis = 1)

submission = pd.DataFrame({
    "id": test_df["id"],
    "target": predicted_classes
})

submission.to_csv("submission.csv", index = False)

print(submission.head())

In [ ]:
print(len(submission))